# 04 — Fiscal calendar

Reads `reference/fiscal_calendar.csv` and `reference/fiscal_quarters.csv` with their provenance record, all produced by `scripts/04_build_fiscal_calendar.py`, and asks whether "next quarter" can be resolved from them.

Section 5.5 of the [annotation guidelines](../docs/annotation-guidelines.md), the window rule, resolves a temporal phrase like "next quarter" into a specific evaluation window against the filer's own fiscal calendar rather than the calendar year. Getting it wrong does not produce an error, it moves a claim onto the adjacent quarter and mislabels every temporal field for that filer, which is why the shape of each calendar is derived and checked rather than assumed.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 220)

REF = PROJECT_ROOT / "reference"
# keep_default_na: most columns here are empty for the ordinary case, and pandas
# would read those as NaN, which breaks every == "" comparison below.
calendar = pd.read_csv(REF / "fiscal_calendar.csv", keep_default_na=False)
quarters = pd.read_csv(REF / "fiscal_quarters.csv", parse_dates=["period_end"])
prov = json.loads((REF / "fiscal_calendar.provenance.json").read_text())

print(f"{len(calendar)} filers, {len(quarters):,} labelled periods, commit {prov['commit']}")
print(f"derived from {', '.join(prov['derived_from'])}, nothing fetched")
print()
for kind, count in prov["counts"]["calendar_types"].items():
    print(f"  {kind:<20} {count:>3} filers")

150 filers, 7,030 labelled periods, commit ed4bd17
derived from reference/filing_dates.csv, reference/filers.csv, nothing fetched

  fixed_date           118 filers
  week_52_53            27 filers
  insufficient_data      5 filers


## Why the declared year end is not enough

EDGAR reports a `fiscalYearEnd` field, one value per filer. It is the obvious place to start and it cannot carry a calendar on its own: for a 52/53-week filer it records whichever date the most recent year happened to land on, and for a filer that changed its year end it records only where it ended up.

In [2]:
week = calendar[calendar["calendar_type"] == "week_52_53"]
ends = quarters[(quarters["quarter"] == 4) & (quarters["cik"].isin(week["cik"]))]

# The spread of month-day values the filer's year end actually took, against the
# single value EDGAR declares for it.
spread = ends.groupby("cik")["period_end"].agg(
    earliest=lambda s: min(s.dt.strftime("%m-%d")),
    latest=lambda s: max(s.dt.strftime("%m-%d")),
    distinct_dates=lambda s: s.dt.strftime("%m-%d").nunique(),
)
spread = spread.join(week.set_index("cik")[["name", "declared_year_end", "year_end_weekday"]])
spread[["name", "declared_year_end", "year_end_weekday", "earliest", "latest", "distinct_dates"]].head(10).reset_index(drop=True)

,name,declared_year_end,year_end_weekday,earliest,latest,distinct_dates
0,APPLIED MATERIALS INC /DE,1026,Sun,10-25,10-31,7
1,AVNET INC,0703,Sat,06-27,07-03,7
2,TARGET CORP,0201,Sat,01-28,02-03,7
3,DOLLAR GENERAL CORP,0129,Fri,01-28,02-03,7
4,INTEL CORP,1227,Sat,12-25,12-31,7
5,KROGER CO,0130,Sat,01-28,02-03,7
6,LOWES COMPANIES INC,0131,Fri,01-28,02-03,7
7,PEPSICO INC,1226,Sat,12-25,12-31,7
8,PUBLIX SUPER MARKETS INC,1226,Sat,12-25,12-31,7
9,SYSCO CORP,0627,Sat,06-27,07-03,7


The declared value names one of those dates and gives no way to tell which. A 52/53-week year end drifts a day or two most years and jumps a week when the 53rd is inserted, so the filer that declares `0926` ends its year anywhere from 24 September to 30 September, and the retailer that declares `0130` crosses from January into February.

The calendar is therefore derived from what filers actually filed. Every 10-K period end is an anchor, and the 10-Q period ends between consecutive anchors are that year's quarters.

## The two shapes, and how they are told apart

In [3]:
annual = quarters[quarters["quarter"] == 4].sort_values(["cik", "period_end"])
gaps = annual.groupby("cik")["period_end"].diff().dt.days.dropna()
labelled = gaps[gaps <= prov["thresholds"]["fiscal_year_max_days"]]

print("days between consecutive annual period ends:")
print(labelled.value_counts().sort_index().to_string())

days between consecutive annual period ends:
period_end
361.0       1
363.0       1
364.0     237
365.0    1000
366.0     344
367.0       1
371.0      51
375.0       1


Four values account for 1,632 of the 1,636 gaps, and they do not overlap. A fixed-date year end lands on the same calendar date, so consecutive years are 365 or 366 days apart. A 52/53-week year ends on the same weekday, which makes it 364 days in an ordinary year and 371 when the extra week is inserted. That separation is what makes the classification decidable rather than a judgement, and it is why a filer is classified on gaps and weekday together rather than on the declared field.

The four stragglers at 361, 363, 367 and 375 days are the whole of the rest of this notebook: two transition years where a filer changed its calendar, and two wrong period ends.

A filer is classified when at least four fifths of its anchors and gaps agree. Below that its year end changed mid-window, which is handled separately rather than averaged.

## Filers whose fiscal year end changed

In [4]:
changed = calendar[calendar["year_end_changed"] == "yes"]
print(f"{len(changed)} of {len(calendar)} filers changed year end inside the window\n")
for item in prov["year_end_changes"]:
    print(f"{item['name'][:34]:<34} {item['from'] or 'unknown':<18} -> {item['to']:<12} at {item['changed_at']}")

3 of 150 filers changed year end inside the window

Archer-Daniels-Midland Co          insufficient_data  -> fixed_date   at 2013-12-31
DEERE & CO                         fixed_date         -> week_52_53   at 2017-10-29
BEST BUY CO INC                    insufficient_data  -> week_52_53   at 2014-02-01


Deere is the case that matters most, because both regimes have enough years to be real. It ran a fixed 31 October year end through fiscal 2016, then moved to a 52/53-week year ending the Sunday nearest the end of October. A claim made in 2015 and a claim made in 2020 resolve "next quarter" against different calendars for the same filer.

Archer-Daniels-Midland moved from a June year end to December in 2012, and Best Buy from a March year end to the Saturday nearest 31 January in 2013. Both changed early enough that the window holds only one anchor under the old regime, which is why the earlier shape reads as `insufficient_data`: one anchor is a date, not a pattern.

The table reports the shape in force after the change and records the earlier one beside it. Calling any of these filers irregular for the whole window would say nothing about how to resolve a quarter under either regime.

In [5]:
deere = quarters[quarters["cik"] == 315189].copy()
deere_annual = deere[deere["quarter"] == 4].sort_values("period_end")
deere_annual["weekday"] = deere_annual["period_end"].dt.day_name().str[:3]
deere_annual["gap_days"] = deere_annual["period_end"].diff().dt.days
deere_annual[["fiscal_year", "period_end", "weekday", "gap_days"]].reset_index(drop=True)

,fiscal_year,period_end,weekday,gap_days
0,2012,2012-10-31,Wed,NaN
1,2013,2013-10-31,Thu,365.0
2,2014,2014-10-31,Fri,365.0
3,2015,2015-10-31,Sat,365.0
4,2016,2016-10-31,Mon,366.0
5,2017,2017-10-29,Sun,363.0
6,2018,2018-10-28,Sun,364.0
7,2019,2019-11-03,Sun,371.0
8,2020,2020-11-01,Sun,364.0
9,2021,2021-10-31,Sun,364.0


## Anchors the derivation does not trust

A period end that disagrees with its neighbours has two possible causes, and they need separating. A filer that changed its year end moves it to a different part of the year. A wrong `reportDate` from EDGAR leaves it roughly where it was and is off by days. Both fail a conformance test; only the first is a change of calendar.

In [6]:
for item in prov["suspect_anchors"]:
    print(f"suspect  {item['name'][:34]:<34} {', '.join(item['anchors'])}")
for item in prov["missing_fiscal_years"]:
    print(f"missing  {item['name'][:34]:<34} fiscal {', '.join(str(y) for y in item['years'])}")

suspect  PUBLIX SUPER MARKETS INC           2013-12-31
suspect  Paramount Global                   2012-12-21
missing  BEST BUY CO INC                    fiscal 2013
missing  GENERAL ELECTRIC CO                fiscal 2014


Paramount is the clearest illustration of why the two are separated. Its first anchor is 21 December 2012, ten days before every other year end it has filed, and December in both cases. Judged on conformance alone that is its first anchor disagreeing, which looks exactly like a year end that changed at the start of the window. Judged on where in the year it falls, it has not moved at all, so it is a wrong period end. Publix is the same failure with a different signature: every one of its year ends is a Saturday except 31 December 2013, a Tuesday.

The missing years are a different defect. General Electric has no annual anchor for fiscal 2014 because the 10-K that would supply it carries the filing date as its period end and was excluded as suspect upstream. Best Buy has none for fiscal 2013 because that year was an eleven-month transition period between the old calendar and the new one.

This is what the `suspect` column in `filing_dates.csv` is for, and it is also its limit. It catches a period end whose filing lag is impossible; it cannot catch one whose lag is plausible but whose date is still wrong. Both Paramount and Publix pass the lag screen and fail here, so the calendar derivation is the second filter.

## Quarter labels

Quarters are labelled by how far through the fiscal year a period end falls, and the observed dates are kept rather than replaced with evenly spaced ones. Nothing assumes a quarter is thirteen weeks.

In [7]:
# A fiscal year whose start is a floor rather than a preceding annual anchor
# cannot be measured: that is each filer's first year in the window, and any year
# following a missing annual report. Both show up as a year exactly as long as the
# floor allows, so they are dropped before measuring quarter position.
FLOOR = prov["thresholds"]["fiscal_year_max_days"]
length = quarters.groupby(["cik", "fiscal_year"])["days_from_year_start"].transform("max")
measurable = quarters[length < FLOOR]

span = measurable.groupby(["cik", "fiscal_year"])["days_from_year_start"].max()
q1 = measurable[measurable["quarter"] == 1]["days_from_year_start"]
print(f"days from year start to the Q1 period end: min {q1.min()}, median {q1.median():.0f}, max {q1.max()}")
print(f"fiscal year lengths: min {span.min()}, median {span.median():.0f}, max {span.max()}")
print()

CHECKS = {320193: ("Apple", 2023), 909832: ("Costco", 2023), 764478: ("Best Buy", 2024), 315189: ("Deere", 2023)}
for cik, (label, year) in CHECKS.items():
    rows = quarters[(quarters["cik"] == cik) & (quarters["fiscal_year"] == year)].sort_values("quarter")
    ends = "  ".join(f"Q{r.quarter} {r.period_end.date()}" for r in rows.itertuples())
    print(f"{label:<10} FY{year}   {ends}")

days from year start to the Q1 period end: min 84, median 90, max 112
fiscal year lengths: min 361, median 365, max 375

Apple      FY2023   Q1 2022-12-31  Q2 2023-04-01  Q3 2023-07-01  Q4 2023-09-30
Costco     FY2023   Q1 2022-11-20  Q2 2023-02-12  Q3 2023-05-07  Q4 2023-09-03
Best Buy   FY2024   Q1 2023-04-29  Q2 2023-07-29  Q3 2023-10-28  Q4 2024-02-03
Deere      FY2023   Q1 2023-01-29  Q2 2023-04-30  Q3 2023-07-30  Q4 2023-10-29


Those four match the quarters the filers themselves report, which is the check the derivation has to pass: the dates come from EDGAR, but the quarter each one closes is inferred here.

The 53-week years are the sharper test, because they are the years an evenly spaced calendar would get wrong.

In [8]:
wk = calendar[calendar["calendar_type"] == "week_52_53"]["cik"]
annual_wk = quarters[(quarters["quarter"] == 4) & (quarters["cik"].isin(wk))].sort_values(["cik", "period_end"])
annual_wk = annual_wk.assign(gap=annual_wk.groupby("cik")["period_end"].diff().dt.days)
long_years = annual_wk[annual_wk["gap"] == 371]
names = calendar.set_index("cik")["name"]

print(f"{len(long_years)} fifty-three week years across {long_years['cik'].nunique()} filers\n")
for cik, group in long_years.groupby("cik"):
    print(f"  {names[cik][:28]:<28} {', '.join(str(d.date()) for d in group['period_end'])}")

51 fifty-three week years across 27 filers

  APPLIED MATERIALS INC /DE    2016-10-30, 2021-10-31
  AVNET INC                    2016-07-02, 2021-07-03
  TARGET CORP                  2013-02-02, 2018-02-03, 2024-02-03
  DOLLAR GENERAL CORP          2017-02-03, 2023-02-03
  INTEL CORP                   2016-12-31, 2022-12-31
  KROGER CO                    2013-02-02, 2018-02-03, 2024-02-03
  LOWES COMPANIES INC          2017-02-03, 2023-02-03
  PEPSICO INC                  2016-12-31, 2022-12-31
  PUBLIX SUPER MARKETS INC     2016-12-31, 2022-12-31
  SYSCO CORP                   2016-07-02, 2021-07-03
  TYSON FOODS, INC.            2015-10-03, 2020-10-03
  JOHNSON & JOHNSON            2016-01-03, 2021-01-03
  DEERE & CO                   2019-11-03
  Apple Inc.                   2017-09-30, 2023-09-30
  HOME DEPOT, INC.             2013-02-03, 2019-02-03
  BEST BUY CO INC              2018-02-03, 2024-02-03
  QUALCOMM INC/DE              2018-09-30, 2024-09-29
  STARBUCKS CORP          

## What is not labelled

In [9]:
for reason, count in prov["counts"]["clean_rows_outside_any_fiscal_year"].items():
    print(f"{count:>3}  {reason.replace('_', ' ')}")
print()
print(f"{prov['counts']['suspect_filing_rows_excluded']:>3}  suspect filing rows excluded before deriving")
print()
insufficient = calendar[calendar["calendar_type"] == "insufficient_data"]
print(f"{len(insufficient)} filers with too few annual anchors to classify:")
print(insufficient[["name", "annual_anchors", "fiscal_years"]].to_string(index=False))

 74  after the last annual anchor
  4  inside a gap between anchors

 10  suspect filing rows excluded before deriving

5 filers with too few annual anchors to classify:
                          name  annual_anchors fiscal_years
           Galaxy Digital Inc.               0             
     Ingram Micro Holding Corp               1    2024-2024
               GE Vernova Inc.               1    2024-2024
Ferguson Enterprises Inc. /DE/               1    2024-2024
       Paramount Skydance Corp               0             


The 74 unlabelled quarters sit after their filer's last annual report, so they belong to a fiscal year the study window never closes. That is an edge effect and expected. The other four sit inside the gaps left by Archer-Daniels-Midland's transition period and Best Buy's missing year, which is a defect in the source rather than an edge.

The five unclassified filers are recent registrants and the two with no filings at all. Three annual anchors is the minimum that can establish a pattern, and guessing below that would put a confident wrong answer in a reference table.

## What this implies for the pilot

**The declared year end cannot be used on its own.** 27 of 150 filers run a 52/53-week calendar whose year end moves every year, and EDGAR's `fiscalYearEnd` reports one date for them. Resolving a quarter from that field would be wrong for 18 per cent of the study set, silently.

**Three filers need their calendar chosen by claim date, not by filer.** Deere, Archer-Daniels-Midland and Best Buy changed year end inside the window. Deere is the one with real exposure on both sides, running a fixed October year end through fiscal 2016 and a 52/53-week year after it.

**The calendar derivation is a second filter on bad period ends.** Paramount and Publix each carry one period end that is wrong but whose filing lag is plausible, so the `suspect` column in `filing_dates.csv` could not see them. Two more anchors are missing outright. Four defects in 1,790 annual periods, all named rather than smoothed.

**`fiscal_year` is a convention, not the filer's own label.** Walmart calls the year ending January 2024 fiscal 2024 and Target calls the year ending January 2023 fiscal 2022, so no single rule matches both. Anything downstream joins on `period_end`.

**Quarter boundaries are observed, not computed.** The Q1 period end falls anywhere from 84 to 112 days into the fiscal year, against a median of 90, and fiscal years themselves run from 361 to 375 days. Placing an evaluation window by counting 91 days from a year start would land on the wrong side of a quarter boundary for a large share of filers.